In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

#### Load Data

In [3]:
dimcity = pd.read_csv("DimCity_cleaned.csv")
dimcustomer = pd.read_csv("DimCustomer_cleaned.csv")
dimdate = pd.read_csv("DimDate_cleaned.csv")
dimemployee = pd.read_csv("DimEmployee_cleaned.csv")
dimstockItem = pd.read_csv("DimStockItem_cleaned.csv")
factsale = pd.read_csv("FactSale_cleaned.csv")

#### Primary Key check

In [4]:
dimcity.columns

Index(['City Key', 'City', 'State Province', 'Country', 'Sales Territory',
       'Location', 'Latest Recorded Population'],
      dtype='object')

In [5]:
assert dimcity['City Key'].is_unique , "City Key is not unique in dimcity"

In [6]:
dimcustomer.columns

Index(['Customer Key', 'WWI Customer ID', 'Customer', 'Bill To Customer',
       'Category', 'Buying Group', 'Primary Contact', 'Postal Code',
       'Credit Limit', 'Valid From', 'Valid To', 'Lineage Key'],
      dtype='object')

In [7]:
assert dimcustomer['Customer Key'].is_unique, "Customer Key is not unique in dimcustomer"

In [8]:
dimemployee.columns

Index(['Employee Key', 'WWI Employee ID', 'Employee', 'Preferred Name',
       'Is Salesperson', 'Valid From', 'Valid To', 'Lineage Key'],
      dtype='object')

In [9]:
dimstockItem.columns

Index(['Stock Item Key', 'WWI Stock Item ID', 'Stock Item', 'Color',
       'Selling Package', 'Buying Package', 'Brand', 'Size', 'Lead Time Days',
       'Quantity Per Outer', 'Is Chiller Stock', 'Barcode', 'Tax Rate',
       'Unit Price', 'Recommended Retail Price', 'Typical Weight Per Unit',
       'Valid From', 'Valid To', 'Lineage Key'],
      dtype='object')

In [10]:
assert dimstockItem['Stock Item Key'].is_unique ,"Stock Item Key is not unique in dimstockItem"

In [11]:
factsale.columns

Index(['Sale Key', 'City Key', 'Customer Key', 'Bill To Customer Key',
       'Stock Item Key', 'Invoice Date Key', 'Delivery Date Key',
       'Salesperson Key', 'WWI Invoice ID', 'Description', 'Package',
       'Quantity', 'Unit Price', 'Tax Rate', 'Total Excluding Tax',
       'Tax Amount', 'Profit', 'Total Including Tax', 'Total Dry Items',
       'Total Chiller Items', 'Lineage Key'],
      dtype='object')

In [12]:
assert factsale['Sale Key'].is_unique, "Sale Key is not unique in factsale"

#### Referential Integrity

In [13]:
invalid_city_keys = factsale[~factsale['City Key'].isin(dimcity['City Key'])]
assert invalid_city_keys.empty, f"{len(invalid_city_keys)} records in factsale have invalid City Keys"

invalid_customer_keys = factsale[~factsale['Customer Key'].isin(dimcustomer['Customer Key'])]
assert invalid_customer_keys.empty, f"{len(invalid_customer_keys)} records in factsale have invalid Customer Key"

invalid_salesperson_keys = factsale[~factsale['Salesperson Key'].isin(dimemployee['Employee Key'])]
assert invalid_salesperson_keys.empty, f"{len(invalid_salesperson_keys)} records in factsale have invalid Salesperson Key"

invalid_stock_item_keys = factsale[~factsale['Stock Item Key'].isin(dimstockItem['Stock Item Key'])]
assert invalid_stock_item_keys.empty, f"{len(invalid_stock_item_keys)} records in factsale have invalid Stock Item Key"

#### duplicate checks

In [14]:
factsale.columns

Index(['Sale Key', 'City Key', 'Customer Key', 'Bill To Customer Key',
       'Stock Item Key', 'Invoice Date Key', 'Delivery Date Key',
       'Salesperson Key', 'WWI Invoice ID', 'Description', 'Package',
       'Quantity', 'Unit Price', 'Tax Rate', 'Total Excluding Tax',
       'Tax Amount', 'Profit', 'Total Including Tax', 'Total Dry Items',
       'Total Chiller Items', 'Lineage Key'],
      dtype='object')

In [15]:
cols = ['City Key', 'Customer Key', 'Bill To Customer Key',
       'Stock Item Key', 'Invoice Date Key', 'Delivery Date Key',
       'Salesperson Key', 'Quantity', 'Unit Price']
factsale.duplicated(subset=cols).sum()

np.int64(0)

In [16]:
factsale.loc[factsale.duplicated(subset=cols, keep=False), cols].sort_values(by=cols)

,City Key,Customer Key,Bill To Customer Key,Stock Item Key,Invoice Date Key,Delivery Date Key,Salesperson Key,Quantity,Unit Price


#### Business Validation

In [17]:
factsale.loc[(factsale['Unit Price'] <= 0) | (factsale['Quantity'] <= 0)]

,Sale Key,City Key,Customer Key,Bill To Customer Key,Stock Item Key,Invoice Date Key,Delivery Date Key,Salesperson Key,WWI Invoice ID,Description,Package,Quantity,Unit Price,Tax Rate,Total Excluding Tax,Tax Amount,Profit,Total Including Tax,Total Dry Items,Total Chiller Items,Lineage Key


In [18]:
factsale['calc_total'] = factsale['Quantity'] * factsale['Unit Price']

factsale[abs(factsale['calc_total'] - factsale['Total Excluding Tax']) > 1]

,Sale Key,City Key,Customer Key,Bill To Customer Key,Stock Item Key,Invoice Date Key,Delivery Date Key,Salesperson Key,WWI Invoice ID,Description,Package,Quantity,Unit Price,Tax Rate,Total Excluding Tax,Tax Amount,Profit,Total Including Tax,Total Dry Items,Total Chiller Items,Lineage Key,calc_total


In [19]:
factsale.loc[abs(factsale['Total Including Tax'] - 
            (factsale['Total Excluding Tax'] + factsale['Tax Amount']))
            > 0.01]

,Sale Key,City Key,Customer Key,Bill To Customer Key,Stock Item Key,Invoice Date Key,Delivery Date Key,Salesperson Key,WWI Invoice ID,Description,Package,Quantity,Unit Price,Tax Rate,Total Excluding Tax,Tax Amount,Profit,Total Including Tax,Total Dry Items,Total Chiller Items,Lineage Key,calc_total


In [20]:
factsale.loc[factsale['Delivery Date Key'] < factsale['Invoice Date Key']]

,Sale Key,City Key,Customer Key,Bill To Customer Key,Stock Item Key,Invoice Date Key,Delivery Date Key,Salesperson Key,WWI Invoice ID,Description,Package,Quantity,Unit Price,Tax Rate,Total Excluding Tax,Tax Amount,Profit,Total Including Tax,Total Dry Items,Total Chiller Items,Lineage Key,calc_total


In [21]:
factsale[
    (factsale['Tax Rate'] < 0) | 
    (factsale['Tax Rate'] > 100)
]

,Sale Key,City Key,Customer Key,Bill To Customer Key,Stock Item Key,Invoice Date Key,Delivery Date Key,Salesperson Key,WWI Invoice ID,Description,Package,Quantity,Unit Price,Tax Rate,Total Excluding Tax,Tax Amount,Profit,Total Including Tax,Total Dry Items,Total Chiller Items,Lineage Key,calc_total


In [22]:
dimdate.head()

,Date,Day Number,Day,Month,Short Month,Calendar Month Number,Calendar Month Label,Calendar Year,Calendar Year Label,Fiscal Month Number,Fiscal Month Label,Fiscal Year,Fiscal Year Label,ISO Week Number
0,2013-01-01,1,1,January,Jan,1,CY2013-Jan,2013,CY2013,3,FY2013-Jan,2013,FY2013,1
1,2013-01-02,2,2,January,Jan,1,CY2013-Jan,2013,CY2013,3,FY2013-Jan,2013,FY2013,1
2,2013-01-03,3,3,January,Jan,1,CY2013-Jan,2013,CY2013,3,FY2013-Jan,2013,FY2013,1
3,2013-01-04,4,4,January,Jan,1,CY2013-Jan,2013,CY2013,3,FY2013-Jan,2013,FY2013,1
4,2013-01-05,5,5,January,Jan,1,CY2013-Jan,2013,CY2013,3,FY2013-Jan,2013,FY2013,1


In [23]:
dimdate.dtypes

Date                     object
Day Number                int64
Day                       int64
Month                    object
Short Month              object
Calendar Month Number     int64
Calendar Month Label     object
Calendar Year             int64
Calendar Year Label      object
Fiscal Month Number       int64
Fiscal Month Label       object
Fiscal Year               int64
Fiscal Year Label        object
ISO Week Number           int64
dtype: object

In [24]:
dimdate['Date'] = pd.to_datetime(dimdate['Date'])

In [25]:
dimdate.dtypes

Date                     datetime64[ns]
Day Number                        int64
Day                               int64
Month                            object
Short Month                      object
Calendar Month Number             int64
Calendar Month Label             object
Calendar Year                     int64
Calendar Year Label              object
Fiscal Month Number               int64
Fiscal Month Label               object
Fiscal Year                       int64
Fiscal Year Label                object
ISO Week Number                   int64
dtype: object

In [26]:
IS_TRUE = True
if IS_TRUE:
    dimdate.to_csv("DimDate_cleaned.csv", index=False)